# Engram: full LoCoMo run with a stronger model

This runs the same frozen Engram version as the overnight Gemma 3 4B test, but with a stronger open model. The comparison shows how much of the loss comes from the memory and how much from the small model.

**Before you start**
1. Choose **Runtime > Change runtime type > T4 GPU** (or any GPU offered).
2. Run the cells in order, top to bottom. Keep this tab open while it runs.
3. Use benchmark data only. Do not put private notes or patient data in this notebook.

Progress is saved to your Google Drive. If the session ends, reopen the notebook and run all the cells again: it continues where it stopped. Free GPU time is limited and can run out, in which case try again later.

## 1. Settings and GPU check

In [ ]:
MODEL = "gemma3:12b"     # about 8 GB. Change this to try another model from ollama.com
TAG = "gemma3_12b"       # name of this run; use a new name for each model
COMMIT = "75de015"       # the frozen Engram version, the same as the overnight run

import subprocess
result = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", shell=True, capture_output=True, text=True)
print("GPU:", result.stdout.strip() or "NONE. Choose Runtime > Change runtime type > GPU, then run this cell again.")

## 2. Connect Google Drive

Google will ask for permission. Results go to a folder called `engram_results`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
SAVE = "/content/drive/MyDrive/engram_results"
os.makedirs(SAVE, exist_ok=True)
print("Saving to", SAVE, "| already there:", os.listdir(SAVE))

## 3. Get the code and benchmark data

In [ ]:
%cd /content
!rm -rf Engram
!git clone -q https://github.com/manjunathhari/Engram.git
%cd /content/Engram
!git checkout -q {COMMIT}
!git clone -q --depth 1 https://github.com/snap-research/locomo.git
!pip install -q bm25s
!git log --oneline | head -1
!ls locomo/data/locomo10.json

## 4. Install the model

This downloads several GB, so it can take a few minutes.

In [ ]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import os, subprocess, time, urllib.request
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1")
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull {MODEL}
!ollama run {MODEL} "Reply with the single word ready."
!ollama ps

## 5. Run the benchmark

1,536 questions, plus 321 date questions answered again without the date fix. It restarts where it stopped. Paste the last line here if it stops with an error.

In [ ]:
import os, shutil, subprocess, sys
os.chdir("/content/Engram")
LOCAL, DRIVE = f"results/locomo_{TAG}.jsonl", f"{SAVE}/locomo_{TAG}.jsonl"
os.makedirs("results", exist_ok=True)
if os.path.exists(DRIVE):
    shutil.copy(DRIVE, LOCAL)
    print("Continuing from", sum(1 for _ in open(LOCAL)), "saved answers")
run = subprocess.Popen([sys.executable, "-u", "chat_kit.py", "locomo", "--url", "http://localhost:11434/v1", "--model", MODEL, "--tag", TAG], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
n = 0
for line in run.stdout:
    if not line.startswith("question "):
        print(line, end="")
        continue
    n += 1
    if n % 25 == 0:
        print(line, end="")
        shutil.copy(LOCAL, DRIVE)
code = run.wait()
if os.path.exists(LOCAL): shutil.copy(LOCAL, DRIVE)
if os.path.exists(f"results/locomo_{TAG}_summary.json"): shutil.copy(f"results/locomo_{TAG}_summary.json", SAVE)
print("Finished" if code == 0 else f"Stopped with an error, code {code}. Run this cell again to continue.")

## 6. Check the results

A finished run has 1,857 answers. Send `locomo_<your tag>.jsonl` from the `engram_results` Drive folder to Claude for blind scoring. The f1 figure is only a rough signal.

In [ ]:
import json
rows = [json.loads(x) for x in open(DRIVE)]
print(len(rows), "answers of 1857")
for system in ("engram", "no_dates"):
    r = [x for x in rows if x["system"] == system]
    if r:
        print(system, "| n", len(r), "| evidence found", round(sum(x["recall"] for x in r) / len(r), 3), "| f1", round(sum(x["f1"] for x in r) / len(r), 3), "| seconds", round(sum(x["secs"] for x in r) / len(r), 1))